## **6° Module**: TT - Design Verifications
* September 22°, 2026
#### ESCOM - IPN:

#### *B.S. in Computer Science*
> Miguel Alexander Sanchez Garcia

#### **0° Introduction**

The resource benchmark of notebook 5 left four questions that its tables could not
answer on their own. This notebook checks each one directly, so the claims that
end up in the technical report rest on something reproducible rather than on
inspection of a summary table.

1. Are conditions **C4 and C5 actually different**? Their complexity rows came out
   identical, which is suspicious.
2. Is the **ansatz necessary**, or could the feature map be measured on its own?
3. **Where are shots configured?** The first shot-sensitivity run produced a flat
   error curve, which is physically impossible.
4. Is there a **cheaper gradient** than parameter-shift, and would a gradient-free
   optimiser such as PSO help?

Each section states the question, runs the check, and closes with the consequence
for the experimental design. Everything runs in seconds except section 4.

**a.** Import the necessary libraries

In [1]:
import time
import warnings
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map, pauli_feature_map, real_amplitudes
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN

warnings.filterwarnings("ignore")
import logging
logging.getLogger("qiskit_machine_learning").setLevel(logging.ERROR)

SEED = 42
rng = np.random.default_rng(SEED)

print("Ready.")

Ready.


#### **1° Are C4 and C5 the same condition?**

**a.** The question

The methodology defines C4 as a ZZFeatureMap and C5 as a PauliFeatureMap with
Pauli terms $(Z, ZZ)$, treating them as independent conditions in order to isolate
the effect of the feature map design.

In the complexity table of notebook 5, all eighteen rows came out **identical**
between the two: same transpiled depth, same CX count, same gate count, at every
$(k, \text{reps})$. Identical cost does not by itself prove identical behaviour,
so the states themselves have to be compared.

The right instrument is the **fidelity** between the two encoded states:

$$F = |\langle \phi_{ZZ}(x) \,|\, \phi_{\text{Pauli}}(x) \rangle|^2$$

$F = 1$ means the states are indistinguishable by any measurement whatsoever.

In [2]:
print("Fidelity between the two encodings, same input vector x")
print()
for k in (4, 8):
    x = rng.uniform(0, np.pi, k)
    s_zz = Statevector.from_instruction(zz_feature_map(k, reps=1).assign_parameters(x))
    s_pl = Statevector.from_instruction(
        pauli_feature_map(k, reps=1, paulis=["Z", "ZZ"]).assign_parameters(x))
    print(f"  k={k}:  |<phi_zz|phi_pauli>|^2 = {abs(s_zz.inner(s_pl)) ** 2:.12f}")

print()
print("  A fidelity of exactly 1 means the two circuits prepare the SAME state.")
print("  In Qiskit the ZZFeatureMap is *defined* as a PauliFeatureMap with")
print("  paulis=['Z','ZZ'], so asking for that Pauli set returns the same circuit.")

Fidelity between the two encodings, same input vector x

  k=4:  |<phi_zz|phi_pauli>|^2 = 1.000000000000
  k=8:  |<phi_zz|phi_pauli>|^2 = 1.000000000000

  A fidelity of exactly 1 means the two circuits prepare the SAME state.
  In Qiskit the ZZFeatureMap is *defined* as a PauliFeatureMap with
  paulis=['Z','ZZ'], so asking for that Pauli set returns the same circuit.


**b.** Candidate replacements for C5

If C5 is to be a genuine second condition it needs a different Pauli set. The
table below reports, for each candidate, the fidelity against the ZZ encoding and
the CX count, which is what the cost depends on.

In [3]:
k, N_DRAWS = 4, 200

# Fidelity depends on the input vector, so a single draw says very little.
# Each candidate is characterised by the distribution over many random inputs.
xs = rng.uniform(0, np.pi, (N_DRAWS, k))
zz_states = [Statevector.from_instruction(zz_feature_map(k, reps=1).assign_parameters(x))
             for x in xs]

rows = []
for paulis in (["Z", "ZZ"], ["Z", "Y", "ZZ"], ["X", "ZZ"], ["Z", "YY"], ["Y", "ZZ"]):
    circ = pauli_feature_map(k, reps=1, paulis=paulis)
    fid = np.array([abs(s_zz.inner(Statevector.from_instruction(circ.assign_parameters(x)))) ** 2
                    for s_zz, x in zip(zz_states, xs)])
    rows.append({"paulis": str(paulis),
                 "fidelity_mean": fid.mean(), "fidelity_std": fid.std(),
                 "fidelity_max": fid.max(),
                 "n_cx": circ.decompose().count_ops().get("cx", 0)})

cand = pd.DataFrame(rows)
print(f"Fidelity against the ZZ encoding over {N_DRAWS} random inputs.")
print("1 means identical; a low mean with a low max means a genuinely")
print("different encoding across the whole input range.")
cand.round(6)

Fidelity against the ZZ encoding over 200 random inputs.
1 means identical; a low mean with a low max means a genuinely
different encoding across the whole input range.


,paulis,fidelity_mean,fidelity_std,fidelity_max,n_cx
0,"['Z', 'ZZ']",1.000000,0.000000,1.000000,12
1,"['Z', 'Y', 'ZZ']",0.331794,0.331844,0.962074,12
2,"['X', 'ZZ']",0.070827,0.133643,0.727172,12
3,"['Z', 'YY']",0.109452,0.129158,0.706488,12
4,"['Y', 'ZZ']",0.314932,0.151062,0.991910,12


**c.** Consequence

As specified, **C4 and C5 are one condition, not two**, and the comparison they
were meant to support measures nothing. C5 needs a Pauli set that is genuinely
different. Several candidates achieve this at an identical CX count, so the second
condition costs nothing extra; which one to adopt changes what the comparison
claims and is therefore a decision for the methodology, not for this notebook.

#### **2° Is the ansatz necessary?**

**a.** The question

Under a precomputed-embedding architecture the variational parameters $\theta$ are
never trained. A natural simplification would be to drop the ansatz altogether and
measure $\langle Z_i \rangle$ directly on $|\phi(x)\rangle$.

There is a reason to doubt that this works. The ZZFeatureMap applies Hadamards
followed by **diagonal** gates only: the $R_Z$ rotations and the
$CX \!-\! R_Z \!-\! CX$ entangling blocks are all diagonal in the computational
basis. A diagonal gate multiplies each amplitude by a phase and therefore cannot
change any amplitude's **magnitude**.

Starting from $H^{\otimes k}|0\rangle$, every amplitude has magnitude
$2^{-k/2}$. If diagonal gates preserve magnitudes, then
$P(\text{qubit}_i = 0) = 1/2$ exactly, for every qubit and every input.

In [4]:
k = 4
obs = [SparsePauliOp.from_sparse_list([("Z", [i], 1.0)], num_qubits=k) for i in range(k)]

print("Feature map ALONE, no ansatz:")
for _ in range(3):
    x = rng.uniform(0, np.pi, k)
    sv = Statevector.from_instruction(zz_feature_map(k, reps=1).assign_parameters(x))
    z = [sv.expectation_value(o).real for o in obs]
    print(f"  x = {np.round(x, 2)}   ->   <Z_i> = {np.round(z, 6)}")

print()
print("Feature map + ansatz with FIXED random theta:")
w = rng.uniform(0, 2 * np.pi, 2 * k)
qc = QuantumCircuit(k)
qc.compose(zz_feature_map(k, reps=1), inplace=True)
qc.compose(real_amplitudes(k, reps=1), inplace=True)
for _ in range(3):
    x = rng.uniform(0, np.pi, k)
    sv = Statevector.from_instruction(qc.assign_parameters(np.concatenate([x, w])))
    z = [sv.expectation_value(o).real for o in obs]
    print(f"  x = {np.round(x, 2)}   ->   <Z_i> = {np.round(z, 4)}")

Feature map ALONE, no ansatz:
  x = [3.12 1.84 0.4  2.82]   ->   <Z_i> = [-0.  0.  0.  0.]
  x = [2.77 1.68 1.95 0.86]   ->   <Z_i> = [-0.  0.  0.  0.]
  x = [0.16 1.87 0.93 2.08]   ->   <Z_i> = [ 0.  0.  0. -0.]

Feature map + ansatz with FIXED random theta:
  x = [1.24 2.13 2.28 1.78]   ->   <Z_i> = [-0.4171  0.085  -0.3782  0.0411]
  x = [2.38 3.09 1.32 1.62]   ->   <Z_i> = [ 0.0467 -0.1821  0.1195  0.2861]
  x = [0.04 2.5  1.63 1.28]   ->   <Z_i> = [ 0.4626 -0.1621  0.0393  0.0477]


**b.** Consequence

Without the ansatz the embedding is **identically zero** and carries no
information at all. The information injected by the feature map lives entirely in
the **phases**, and a measurement in the $Z$ basis is blind to phases.

Three implications follow:

1. The ansatz cannot be dropped. Under a precomputed architecture $\theta$ must be
   **fixed with a declared seed**, not absent.
2. The particular fixed $\theta$ decides which projection of the phase information
   becomes visible, so it is a reportable parameter of the experiment.
3. The **fidelity kernel does not share this limitation**, because it compares full
   states, phases included. Kernel-based metrics (KTA, geometric difference) and
   $\langle Z \rangle$-based metrics (Davies-Bouldin, Fisher, and the classifier)
   are therefore measuring genuinely different objects, and may disagree.

#### **3° Where are shots configured?**

**a.** The question

The first shot-sensitivity run reported an error that barely moved between 512 and
8192 shots. That is impossible: the standard error of an estimated expectation
value decays as $1/\sqrt{n}$, so a sixteenfold increase in shots must reduce it
about fourfold.

The suspicion is that the shot count was never actually changing. The check below
sets the target precision in the two places where it plausibly belongs and
measures the spread of repeated evaluations, which is the quantity that must
follow $1/\sqrt{n}$.

In [5]:
k = 4
fm, ans = zz_feature_map(k, reps=1), real_amplitudes(k, reps=1)
qc = QuantumCircuit(k)
qc.compose(fm, inplace=True)
qc.compose(ans, inplace=True)
obs = [SparsePauliOp.from_sparse_list([("Z", [i], 1.0)], num_qubits=k) for i in range(k)]
X = rng.uniform(0, np.pi, (1, k))
w = rng.uniform(0, 2 * np.pi, 2 * k)


def spread(shots, where):
    """Standard deviation of repeated forward passes, averaged over the k outputs."""
    prec = 1.0 / np.sqrt(shots)
    if where == "estimator":
        est, kw = AerEstimator(options={"default_precision": prec}), {}
    else:
        est, kw = AerEstimator(), {"default_precision": prec}
    qnn = EstimatorQNN(circuit=qc, input_params=list(fm.parameters),
                       weight_params=list(ans.parameters), observables=obs,
                       estimator=est, input_gradients=False, **kw)
    runs = np.array([qnn.forward(X, w)[0] for _ in range(20)])
    return runs.std(axis=0).mean()


print(f"{'shots':>7} {'on the estimator':>18} {'on the QNN':>12} {'expected':>10}")
for shots in (256, 1024, 8192, 65536):
    print(f"{shots:7d} {spread(shots, 'estimator'):18.5f} "
          f"{spread(shots, 'qnn'):12.5f} {1 / np.sqrt(shots):10.5f}")

  shots   on the estimator   on the QNN   expected
    256            0.01425      0.06082    0.06250


   1024            0.01433      0.02736    0.03125
   8192            0.01476      0.00932    0.01105


  65536            0.01398      0.00381    0.00391


**b.** Consequence

Setting the precision **on the estimator has no effect**; setting it **on the
`EstimatorQNN`** tracks $1/\sqrt{n}$ as it must. The cause is visible in the
source: `EstimatorQNN._forward` calls

```python
job = self.estimator.run(circuit_observable_params, precision=self._default_precision)
```

which passes the QNN's own precision and overrides whatever the estimator was
configured with.

This is worth separating carefully. That measurement precision improves as
$1/\sqrt{n}$ is a property of **quantum measurement statistics**. That the setting
must go in one particular constructor and is silently ignored in the other is a
property of **this library's implementation**. Only the first belongs in the
theoretical chapter of the report; the second belongs in the implementation
section, and is exactly the kind of detail that makes results irreproducible when
left undocumented.

#### **4° Is there a cheaper gradient?**

**a.** The question

Parameter-shift is exact, but it costs $2$ circuit evaluations **per parameter**,
whereas classical backpropagation obtains the whole gradient in a cost independent
of the parameter count. That asymmetry is what makes joint training unaffordable.

`qiskit-machine-learning` ships three gradient estimators. **Lin-comb** is exact
and uses an auxiliary qubit; **SPSA** approximates the gradient with a fixed number
of evaluations regardless of how many parameters there are, trading accuracy for
cost.

> This cell is the slow one, and one of the methods does not finish in any
> reasonable time. Each measurement therefore runs under a wall-clock budget:
> exceeding it is recorded as a result, not as an error, since a gradient that
> slow cannot train 2,864 samples for 50 epochs regardless of its exact cost.

In [6]:
import signal
from qiskit_machine_learning.gradients import (ParamShiftEstimatorGradient,
                                                LinCombEstimatorGradient,
                                                SPSAEstimatorGradient)


class Budget(Exception):
    """Raised when a single measurement exceeds its wall-clock budget."""


def _ring(signum, frame):
    raise Budget()


K_GRAD = [4, 8]          # add 12 only if you are willing to wait
BUDGET_S = 90            # abort any single measurement past this; see note below
N_TRAIN_TOTAL = 2864     # mass 1318 + calcification 1546
EPOCHS = 50
est = StatevectorEstimator()


def backward_cost(k, reps, grad):
    """Seconds per sample for one backward pass."""
    fm, ans = zz_feature_map(k, reps=1), real_amplitudes(k, reps=reps)
    circ = QuantumCircuit(k)
    circ.compose(fm, inplace=True)
    circ.compose(ans, inplace=True)
    ob = [SparsePauliOp.from_sparse_list([("Z", [i], 1.0)], num_qubits=k) for i in range(k)]
    qnn = EstimatorQNN(circuit=circ, input_params=list(fm.parameters),
                       weight_params=list(ans.parameters), observables=ob,
                       estimator=est, input_gradients=False, gradient=grad)
    Xb = rng.uniform(0, np.pi, (2, k))
    wb = rng.uniform(0, 2 * np.pi, qnn.num_weights)
    qnn.forward(Xb[:1], wb)
    # A measurement that does not finish inside the budget is itself the result:
    # a gradient that slow cannot train 2,864 samples for 50 epochs.
    signal.signal(signal.SIGALRM, _ring)
    signal.alarm(BUDGET_S)
    try:
        t0 = time.perf_counter()
        qnn.backward(Xb, wb)
        return (time.perf_counter() - t0) / 2
    finally:
        signal.alarm(0)


rows = []
for k in K_GRAD:
    for name, grad in [("parameter-shift", ParamShiftEstimatorGradient(est)),
                       ("lin-comb", LinCombEstimatorGradient(est)),
                       ("SPSA (approximate)", SPSAEstimatorGradient(est, epsilon=0.01, batch_size=1))]:
        t_start = time.perf_counter()
        try:
            dt = backward_cost(k, 1, grad)
            rows.append({"k": k, "method": name, "s_per_sample": dt,
                         "hours_50_epochs": dt * N_TRAIN_TOTAL * EPOCHS / 3600})
            print(f"  k={k:2d}  {name:20s} {dt * 1000:9.1f} ms/sample  "
                  f"->  {dt * N_TRAIN_TOTAL * EPOCHS / 3600:8.2f} h")
        except Exception as exc:
            # The alarm fires inside the estimator, which wraps it in its own
            # error, so the elapsed time is what tells the two cases apart.
            elapsed = time.perf_counter() - t_start
            if isinstance(exc, Budget) or elapsed >= BUDGET_S * 0.9:
                rows.append({"k": k, "method": name, "s_per_sample": np.nan,
                             "hours_50_epochs": np.nan})
                print(f"  k={k:2d}  {name:20s} over the {BUDGET_S}s budget "
                      f"(>{BUDGET_S / 2 * 1000:.0f} ms/sample) -> unusable")
            else:
                print(f"  k={k:2d}  {name:20s} FAILED after {elapsed:.1f}s: "
                      f"{type(exc).__name__}: {str(exc)[:50]}")

gradients = pd.DataFrame(rows)
print()
print("Hours are for 50 epochs over both subsets, one quantum condition.")

  k= 4  parameter-shift           55.9 ms/sample  ->      2.22 h


  k= 4  lin-comb                 241.6 ms/sample  ->      9.61 h
  k= 4  SPSA (approximate)         6.9 ms/sample  ->      0.27 h


  k= 8  parameter-shift          554.1 ms/sample  ->     22.04 h


  k= 8  lin-comb             over the 90s budget (>45000 ms/sample) -> unusable


  k= 8  SPSA (approximate)       161.7 ms/sample  ->      6.43 h

Hours are for 50 epochs over both subsets, one quantum condition.


**b.** Gradient-free optimisation: what PSO would cost

Particle Swarm Optimisation uses no derivatives. Its cost per iteration is
*swarm size* $\times$ *one forward pass over the data*, so it is priced from the
forward timings already measured in notebook 5 rather than re-measured here.

In [7]:
bench = pd.read_csv("/Users/alexander_san/TT/Code/results/5_benchmark_resultados.csv")
fwd = bench[(bench.feature_map == "zz") & (bench.reps == 1)].set_index("k")

SWARM, ITERS, BATCH = 30, 200, 128
evals = SWARM * ITERS

rows = []
for k in fwd.index:
    f = fwd.loc[k, "fwd_per_sample_s"]
    rows.append({
        "k": k,
        "params": int(fwd.loc[k, "n_weights"]),
        "pso_full_h": f * 1318 * evals / 3600,
        "pso_batch128_h": f * BATCH * evals / 3600,
        "param_shift_h": fwd.loc[k, "total_both_h"] / 2,
    })

pso = pd.DataFrame(rows)
print(f"PSO with {SWARM} particles x {ITERS} iterations = {evals} objective evaluations")
print("All columns are hours for the mass subset only.")
pso.round(1)

PSO with 30 particles x 200 iterations = 6000 objective evaluations
All columns are hours for the mass subset only.


,k,params,pso_full_h,pso_batch128_h,param_shift_h
0,8,16,36.4,3.5,12.0
1,12,24,205.3,19.9,95.4
2,16,32,2320.6,225.4,1326.0


**c.** Consequence

PSO evaluated on the full training set is **worse** than parameter-shift, because
it pays thousands of passes over all the data while parameter-shift pays only
$2p$ evaluations per sample and $p$ is small. With mini-batches it becomes several
times cheaper, but still an order of magnitude more expensive than not training
$\theta$ at all.

More importantly, changing optimiser does not avoid the underlying obstacle.
Where a cost landscape exhibits a **barren plateau**, the cost *differences*
between nearby points are themselves exponentially small, so a swarm comparing
particle scores is moving on noise rather than signal. Gradient-free methods do
not escape this.

> Reference to verify before citing: Arrasmith, Cerezo, Czarnik, Cincio and Coles,
> *Effect of barren plateaus on gradient-free optimization*, Quantum 5, 558 (2021).

What PSO would genuinely buy is the ability to optimise a **non-differentiable**
objective directly, such as AUC-ROC, kernel-target alignment, or the Fisher ratio.
None of the separability metrics used in this work are naturally differentiable,
so that capability has no substitute among gradient-based methods.

#### **5° Summary**

In [8]:
print("=" * 66)
print("DESIGN VERIFICATIONS - SUMMARY")
print("=" * 66)
print("  1. C4 and C5 are the SAME condition as specified.")
print("     pauli_feature_map(paulis=['Z','ZZ']) == zz_feature_map, fidelity 1.0")
print("     -> C5 requires a different Pauli set; several cost the same.")
print()
print("  2. The ansatz is NOT optional.")
print("     Without it <Z_i> = 0 for every input: the feature map writes its")
print("     information into phases, and Z measurements cannot see phases.")
print("     -> theta must be fixed with a declared seed, not removed.")
print()
print("  3. Shot precision must be set on the EstimatorQNN, not the estimator.")
print("     Setting it on the estimator is silently ignored.")
print("     -> an implementation detail, but one that invalidates results.")
print()
print("  4. SPSA is by far the cheapest gradient, around ten times faster than")
print("     parameter-shift; lin-comb is several times SLOWER and unusable at k=8.")
print("     Even so, no gradient method makes joint training competitive with a")
print("     precomputed embedding, and SPSA only approximates the gradient, so it")
print("     needs more iterations to converge than the raw speedup suggests.")
print("     PSO's real value is optimising non-differentiable objectives.")
print("=" * 66)

DESIGN VERIFICATIONS - SUMMARY
  1. C4 and C5 are the SAME condition as specified.
     pauli_feature_map(paulis=['Z','ZZ']) == zz_feature_map, fidelity 1.0
     -> C5 requires a different Pauli set; several cost the same.

  2. The ansatz is NOT optional.
     Without it <Z_i> = 0 for every input: the feature map writes its
     information into phases, and Z measurements cannot see phases.
     -> theta must be fixed with a declared seed, not removed.

  3. Shot precision must be set on the EstimatorQNN, not the estimator.
     Setting it on the estimator is silently ignored.
     -> an implementation detail, but one that invalidates results.

  4. SPSA is by far the cheapest gradient, around ten times faster than
     parameter-shift; lin-comb is several times SLOWER and unusable at k=8.
     Even so, no gradient method makes joint training competitive with a
     precomputed embedding, and SPSA only approximates the gradient, so it
     needs more iterations to converge than the ra